In [1]:
!pip install pandas==2.3.3
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 107.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4

# DATA PREP

In [2]:
# ============================================================
# DATA PREP — PHYSIONET 2017 (DROP-IN REPLACEMENT)
# ============================================================

import os
import numpy as np
import pandas as pd
import wfdb
from collections import Counter
from scipy.signal import resample

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)

FS = 300
TARGET_FS = 300
SEG_LEN = 9000

ROOT_DIR = "/kaggle/input/physionet-af-dataset-modified/PhysioNet_AF_Dataset_modified/physionet.org/files/challenge-2017/1.0.0"
TRAIN_DIR = os.path.join(ROOT_DIR, "training")
VAL_DIR   = os.path.join(ROOT_DIR, "validation")
TRAIN_REF = os.path.join(TRAIN_DIR, "REFERENCE.csv")
VAL_REF   = os.path.join(VAL_DIR, "REFERENCE.csv")

def read_physionet_2017_records(base_dir, ref_csv):
    ref = pd.read_csv(ref_csv, header=None, names=["record", "label"])
    label_map = {"N": "N", "A": "A", "O": None, "~": None}

    Signals, Labels, Ids = [], [], []
    for _, row in ref.iterrows():
        rec = row["record"]
        lbl = label_map.get(row["label"])
        if lbl is None:
            continue
        try:
            record = wfdb.rdrecord(os.path.join(base_dir, rec))
            sig = record.p_signal[:, 0].astype(np.float32)
            sig = resample(sig, int(len(sig) * TARGET_FS / FS))
            if len(sig) >= SEG_LEN:
                Signals.append(sig)
                Labels.append(lbl)
                Ids.append(rec)
        except:
            continue
    return Signals, pd.Series(Labels, dtype="category"), np.array(Ids)

Signals_tr, Labels_tr, Ids_tr = read_physionet_2017_records(TRAIN_DIR, TRAIN_REF)
Signals_va, Labels_va, Ids_va = read_physionet_2017_records(VAL_DIR, VAL_REF)

Signals = Signals_tr + Signals_va
Labels  = pd.concat([Labels_tr, Labels_va], ignore_index=True)
Ids     = np.concatenate([Ids_tr, Ids_va])

records_df = pd.DataFrame({"id": Ids, "signal": Signals, "label": Labels})
print("Label distribution:", Counter(Labels))

train_ids, test_ids = train_test_split(
    records_df["id"].unique(), test_size=0.2, random_state=42
)

train_df = records_df[records_df["id"].isin(train_ids)].reset_index(drop=True)
test_df  = records_df[records_df["id"].isin(test_ids)].reset_index(drop=True)

def segment_signals_from_df(df, seg_len=SEG_LEN):
    X, Y = [], []
    for _, row in df.iterrows():
        x, y = row["signal"], row["label"]
        for i in range(len(x) // seg_len):
            X.append(x[i*seg_len:(i+1)*seg_len])
            Y.append(y)
    return X, Y

XTest, YTest = segment_signals_from_df(test_df)
le_test = LabelEncoder()
YTest_enc = le_test.fit_transform(YTest)

2026-02-02 10:18:48.314803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770027528.536407      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770027528.602388      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770027529.146735      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770027529.146781      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770027529.146784      24 computation_placer.cc:177] computation placer alr

Label distribution: Counter({'N': 4686, 'A': 688})


# MODEL 3 — CWT + U-NET (REG + DROPOUT)

In [3]:
# ============================================================
# MODEL 3: CWT + U-NET — REG + DROPOUT (MEMORY-SAFE)
# ============================================================

import pywt
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.utils import resample as sk_resample
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    Dense, Dropout, Flatten, BatchNormalization,
    GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

tf.config.optimizer.set_jit(False)

# -----------------------------
# REDUCED CWT RESOLUTION
# -----------------------------
SCALES = np.arange(1, 65)   # ↓ from 128 → 64
TIME_DOWNSAMPLE = 2         # ↓ 9000 → 4500

def cwt_image(x):
    x = x[::TIME_DOWNSAMPLE]
    c, _ = pywt.cwt(x, SCALES, "morl")
    return np.log1p(np.abs(c) ** 2).astype(np.float32)

# -----------------------------
# DATA
# -----------------------------
Xtr_raw, Ytr_raw = segment_signals_from_df(train_df)
Xte_raw, Yte_raw = segment_signals_from_df(test_df)

# -----------------------------
# OVERSAMPLE INDICES (NOT SIGNALS)
# -----------------------------
indices = np.arange(len(Ytr_raw))
counter = Counter(Ytr_raw)
max_count = max(counter.values())

os_indices = []

for cls in counter:
    cls_idx = indices[np.array(Ytr_raw) == cls]
    res_idx = sk_resample(
        cls_idx,
        replace=True,
        n_samples=max_count,
        random_state=42
    )
    os_indices.extend(res_idx)

# Shuffle oversampled indices
os_indices = np.array(os_indices)
np.random.shuffle(os_indices)

# -----------------------------
# CWT COMPUTATION (NO STACKING)
# -----------------------------
n_train = len(os_indices)
n_test  = len(Xte_raw)

# Infer CWT shape from first sample
tmp = cwt_image(Xtr_raw[0])
H, W = tmp.shape

Xtr = np.empty((n_train, H, W, 1), dtype=np.float32)
Xte = np.empty((n_test,  H, W, 1), dtype=np.float32)

for i, idx in enumerate(os_indices):
    Xtr[i, ..., 0] = cwt_image(Xtr_raw[idx])

for i in range(n_test):
    Xte[i, ..., 0] = cwt_image(Xte_raw[i])

# -----------------------------
# NORMALIZATION
# -----------------------------
mean, std = Xtr.mean(), Xtr.std() + 1e-8
Xtr = (Xtr - mean) / std
Xte = (Xte - mean) / std

# -----------------------------
# LABELS
# -----------------------------
le = LabelEncoder()
Ytr_enc = le.fit_transform(np.array(Ytr_raw)[os_indices])
Yte_enc = le.transform(Yte_raw)

# -----------------------------
# Model
# -----------------------------
def build_cwt_model(inp_shape):
    inp = Input(shape=inp_shape)

    x = Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 4))(x)
    
    x = Conv2D(64, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 4))(x)
    
    x = Conv2D(8, 1, activation="relu")(x)   # bottleneck
    x = MaxPooling2D(pool_size=(2, 4))(x)
    
    x = Flatten()(x)
    x = Dense(256, activation="relu")(x)

    out = Dense(2, activation="softmax", dtype="float32")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_cwt = build_cwt_model(Xtr.shape[1:])

# -----------------------------
# Training (integer labels)
# -----------------------------
model_cwt.fit(
    Xtr,
    Ytr_enc,
    epochs=30,
    batch_size=16,
    verbose=0
)

# -----------------------------
# Evaluation
# -----------------------------
y_prob_cwt = model_cwt.predict(Xte)[:, 1]
y_pred_cwt = (y_prob_cwt >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(Yte_enc, y_pred_cwt).ravel()

print("\nCWT RESULTS")
print("ACC :", accuracy_score(Yte_enc, y_pred_cwt))
print("PREC:", precision_score(Yte_enc, y_pred_cwt))
print("REC :", recall_score(Yte_enc, y_pred_cwt))
print("SPEC:", tn / (tn + fp))
print("F1  :", f1_score(Yte_enc, y_pred_cwt))
print("AUC :", roc_auc_score(Yte_enc, y_prob_cwt))

np.save("y_prob_cwt.npy", y_prob_cwt)
np.save("cwt_mean.npy", mean)
np.save("cwt_std.npy", std)
# Save CWT U-Net model
model_cwt.save("cwt_unet_reg_dropout.h5")
print("CWT U-Net model saved.")

I0000 00:00:1770027842.119524      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1770027842.125321      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1770027866.555876      83 service.cc:152] XLA service 0x7ef6e01117b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770027866.556925      83 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1770027866.556938      83 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1770027867.158297      83 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1770027872.698534      83 device_compiler.h:188] Compiled clust

38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 57ms/step

CWT RESULTS
ACC : 0.8983193277310925
PREC: 0.931665062560154
REC : 0.9508840864440079
SPEC: 0.5872093023255814
F1  : 0.9411764705882353


AUC : 0.8909483940238497
CWT U-Net model saved.
